# BM25 を自分で実装してパラメータの働きを確かめる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_search/bm25_scratch.ipynb)

NLP クックブック 18-02 ／ 検索

キーワード検索の標準的なスコアである BM25 が、何を計算しているのかを理解したい。パラメータ k1 と b を変えると何が起きるかを見たい。

In [ ]:
!pip install -q rank_bm25 fugashi unidic-lite

In [ ]:
import math
from collections import Counter
import numpy as np
from fugashi import Tagger

tagger = Tagger()
tok = lambda s: [w.surface for w in tagger(s) if w.feature.pos1 not in ("助詞", "助動詞", "補助記号")]

docs = [
    "猫は可愛い。",
    "猫と犬はどちらも人気のあるペットで、猫は室内で、犬は散歩が必要なので屋外でも飼われることが多い。",
    "猫、猫、猫。猫が大好きだ。",
    "犬の散歩は毎日の習慣である。",
    "ペットを飼うと生活が豊かになると言われている。",
]
tokenized = [tok(d) for d in docs]
N, avgdl = len(docs), np.mean([len(d) for d in tokenized])
df = Counter(t for d in tokenized for t in set(d))

def bm25(query, k1=1.5, b=0.75):
    scores = []
    for d in tokenized:
        tf, total = Counter(d), 0.0
        for t in tok(query):
            if t not in tf:
                continue
            idf = math.log((N - df[t] + 0.5) / (df[t] + 0.5) + 1)           # 珍しい語ほど重い
            norm = tf[t] * (k1 + 1) / (tf[t] + k1 * (1 - b + b * len(d) / avgdl))   # 出現回数は頭打ち、長い文書は割り引く
            total += idf * norm
        scores.append(total)
    return np.array(scores)

for q in ["猫", "猫 ペット"]:
    print(q, [round(s, 2) for s in bm25(q)])

In [ ]:
# k1: 出現回数をどこまで重視するか / b: 文書の長さでどこまで割り引くか
print("文書の長さ:", [len(d) for d in tokenized])
for k1, b in [(1.5, 0.75), (0.0, 0.75), (10.0, 0.75), (1.5, 0.0), (1.5, 1.0)]:
    s = bm25("猫", k1, b)
    print(f"k1={k1:4} b={b:4}  スコア {[round(x, 2) for x in s[:3]]}  順位 {list(np.argsort(-s)[:3])}")

In [ ]:
# ライブラリの実装と順位が一致することを確かめる (IDF の式に細かい違いがあるので値は少し違う)
from rank_bm25 import BM25Okapi

lib = BM25Okapi(tokenized, k1=1.5, b=0.75).get_scores(tok("猫 ペット"))
mine = bm25("猫 ペット")
print([round(x, 2) for x in lib], list(np.argsort(-lib)))
print([round(x, 2) for x in mine], list(np.argsort(-mine)))

- BM25 = 「語の珍しさ (IDF)」×「その文書での出現回数 (頭打ちあり)」を、クエリの語について足したもの。
- k1 を 0 にすると出現回数を無視する (含むか含まないかだけ)。大きくすると、同じ語を繰り返す文書ほど高くなる。
- b を 0 にすると文書の長さを無視し、1 にすると長さに比例して割り引く。長い文書は語を多く含みやすいので、その有利さを打ち消している。
- 既定値 (k1=1.2〜2.0、b=0.75) は多くの場合そのままでよい。タイトルのような短い文書ばかりなら b を小さくする。